# SpatialMesh — Day 9: GNN Training Data Generation
**Goal:** Generate thousands of graph snapshots from synthetic 4-speaker conversational scenes.

**Critical:** each clip is HRTF-convolved at a position BEFORE CNN embedding, so embeddings carry real spatial info. Raw mono → meaningless embeddings.

**Pipeline per scene:** pick 4 clips → assign init positions → HRTF convolve → CNN embed → randomize conversational behavior → build graph features → save snapshot.

**No labels.** The loss teaches positions on Day 10. We only generate conversational scenarios.

In [ ]:
# Cell 1 — Setup
!pip install -q soundfile librosa python-sofa 2>/dev/null
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, soundfile as sf, librosa
from pathlib import Path
from scipy.signal import fftconvolve
import random, json, time, os
from multiprocessing import Pool, cpu_count
from google.colab import drive
drive.mount('/content/drive')

BASE      = '/content/drive/MyDrive/SpatialMesh/'
AUDIO_DIR = BASE + 'Libri Speech/'
SOFA_DIR  = BASE + 'sofa/'
CNN_PATH  = BASE + 'models/cnn_encoder_best.pt'
OUT_DIR   = BASE + 'gnn_data/'
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

# Constants
N_SPK=4; SR=48000; SEG=3.0; NODE_DIM=133; EDGE_DIM=7
ALPHA_SIG=5.0; OVERLAP_WIN=5.0
print('CPUs:', cpu_count())

In [ ]:
# Cell 2 — CNN encoder (exact Day 5 architecture)
class CNNEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv1d(2,64,64,4,30),   nn.BatchNorm1d(64),  nn.ReLU(),
            nn.Conv1d(64,128,32,2,15), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128,256,16,2,7), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Conv1d(256,128,8,2,3),  nn.BatchNorm1d(128),
        )
    def forward(self,x):
        return self.backbone(x).mean(dim=-1)

# CPU for multiprocessing (each worker loads its own copy)
cnn = CNNEncoder()
cnn.load_state_dict(torch.load(CNN_PATH, map_location='cpu'))
cnn.eval()
print('CNN loaded.')

In [ ]:
# Cell 3 — Load SOFA HRTFs into memory (small: P0001-P0018)
import sofa

def load_hrtf(sofa_path):
    """Returns dict: positions [N,2] (az,el deg), irs [N,2,taps]."""
    hf = sofa.Database.open(sofa_path)
    src = hf.Source.Position.get_values(system='spherical')  # [N,3] az,el,r
    irs = hf.Data.IR.get_values()  # [N, 2(receivers), taps]
    hf.close()
    pos = src[:, :2]  # az, el
    return {'pos': pos.astype(np.float32), 'irs': irs.astype(np.float32)}

sofa_files = sorted(Path(SOFA_DIR).glob('*.sofa'))
print(f'Found {len(sofa_files)} SOFA files')
# Preload all (small count) — workers index into this
HRTFS = [load_hrtf(str(p)) for p in sofa_files]
print(f'Loaded {len(HRTFS)} HRTF subjects. '
      f'First subject positions: {HRTFS[0]["pos"].shape}')

In [ ]:
# Cell 4 — Audio loading + HRTF convolution helpers
AUDIO_FILES = sorted(Path(AUDIO_DIR).rglob('*.flac')) + \
              sorted(Path(AUDIO_DIR).rglob('*.wav'))
print(f'Found {len(AUDIO_FILES)} audio clips')

def load_clip(path, sr=SR, dur=SEG):
    a, osr = sf.read(str(path))
    if a.ndim>1: a=a[:,0]
    if osr!=sr: a=librosa.resample(a, orig_sr=osr, target_sr=sr)
    n=int(dur*sr)
    a = a[:n] if len(a)>=n else np.pad(a,(0,n-len(a)))
    return a.astype(np.float32)

def nearest_hrtf_idx(hrtf, az_deg, el_deg):
    # az in [0,360) in SOFA; convert our [-180,180] → [0,360)
    az = az_deg % 360
    d = np.abs(hrtf['pos'][:,0]-az) + np.abs(hrtf['pos'][:,1]-el_deg)
    return int(np.argmin(d))

def hrtf_convolve(mono, hrtf, az_deg, el_deg):
    """Convolve mono → binaural stereo at (az,el). Returns [2, samples]."""
    idx = nearest_hrtf_idx(hrtf, az_deg, el_deg)
    ir_L = hrtf['irs'][idx,0]; ir_R = hrtf['irs'][idx,1]
    L = fftconvolve(mono, ir_L)[:len(mono)]
    R = fftconvolve(mono, ir_R)[:len(mono)]
    return np.stack([L,R], axis=0).astype(np.float32)

In [ ]:
# ------------------------------------------------------------
# Cell 4b — Group audio files by speaker (NEW)
# LibriSpeech path structure: .../speaker_id/chapter/utt.flac
# We group by speaker so we can force "same speaker, diff clip"
# hard cases (most confusable) and "similar voice" cases.
# ------------------------------------------------------------
from collections import defaultdict

def group_by_speaker(audio_files):
    """Returns dict: speaker_id -> list of file paths."""
    groups = defaultdict(list)
    for f in audio_files:
        # LibriSpeech: speaker id is the parent-parent folder name,
        # or the leading number in the filename (e.g. 61-70968-0000.flac)
        name = f.stem
        spk = name.split('-')[0] if '-' in name else f.parent.parent.name
        groups[spk].append(f)
    return dict(groups)

SPEAKER_GROUPS = group_by_speaker(AUDIO_FILES)
SPEAKER_IDS = list(SPEAKER_GROUPS.keys())
print(f'{len(SPEAKER_IDS)} speakers, '
      f'{sum(len(v) for v in SPEAKER_GROUPS.values())} clips total')
for s in SPEAKER_IDS[:8]:
    print(f'  speaker {s}: {len(SPEAKER_GROUPS[s])} clips')

In [ ]:
# ------------------------------------------------------------
# Cell 5 — Scene sampler (UPDATED: fixes 2, 3, 8)
# ------------------------------------------------------------
EPS = 1e-8

def random_init_positions(n=N_SPK, max_tries=20):
    """Fix 3: fully random init layout, reject near-collisions.
    Prevents the GNN memorizing a fixed geometry template."""
    for _ in range(max_tries):
        az = np.random.uniform(-1, 1, n)
        el = np.random.uniform(-0.3, 0.3, n)
        ok = True
        for i in range(n):
            for j in range(i+1, n):
                if abs(az[i]-az[j]) < 0.1 and abs(el[i]-el[j]) < 0.1:
                    ok = False
        if ok:
            return np.stack([az, el], axis=1).astype(np.float32)
    # fallback: spread
    return np.stack([np.array([-0.7,-0.2,0.3,0.8]),
                     np.zeros(n)], axis=1).astype(np.float32)


def pick_clips_for_scene():
    """Fix 2: 35% hard cases (similar/same speaker), 65% random.
    Returns (clip_paths[4], speaker_ids[4], hard_case_bool)."""
    hard = random.random() < 0.35

    if hard and len(SPEAKER_IDS) >= 1:
        mode = random.random()
        if mode < 0.5:
            # SAME speaker, different clips — hardest case
            spk = random.choice(
                [s for s in SPEAKER_IDS if len(SPEAKER_GROUPS[s]) >= 2]
                or SPEAKER_IDS)
            pool = SPEAKER_GROUPS[spk]
            clips = [random.choice(pool) for _ in range(N_SPK)]
            spk_ids = [spk]*N_SPK
        else:
            # TWO speakers split across 4 slots — similar-voice confusion
            two = random.sample(SPEAKER_IDS, min(2, len(SPEAKER_IDS)))
            if len(two) == 1: two = two*2
            clips, spk_ids = [], []
            for k in range(N_SPK):
                s = two[k % 2]
                clips.append(random.choice(SPEAKER_GROUPS[s]))
                spk_ids.append(s)
    else:
        # random 4 distinct speakers (or clips if few speakers)
        if len(SPEAKER_IDS) >= N_SPK:
            chosen = random.sample(SPEAKER_IDS, N_SPK)
            clips = [random.choice(SPEAKER_GROUPS[s]) for s in chosen]
            spk_ids = chosen
        else:
            clips = random.sample(AUDIO_FILES, N_SPK)
            spk_ids = ['mix']*N_SPK

    return clips, spk_ids, hard


def sample_scene_config():
    """Conversational geometry. Fix 8: speaker dropout via n_active."""
    # Fix 8: vary active count — sometimes mute speakers
    n_active = random.choices([2,3,4], weights=[0.25,0.40,0.35])[0]
    active = sorted(random.sample(range(N_SPK), n_active))
    activity = np.zeros(N_SPK, dtype=np.float32); activity[active] = 1.0

    # Dominance — sometimes one strongly dominant
    dominance = np.random.rand(N_SPK).astype(np.float32) * 0.3
    dominance[activity == 0] = 0.0
    if random.random() < 0.5 and active:
        dominance[random.choice(active)] = random.uniform(0.7, 1.0)
    dominance = dominance / (dominance.max() + EPS)

    # Overlap matrix — active pairs only
    overlap = np.zeros((N_SPK, N_SPK), dtype=np.float32)
    for a in active:
        for b in active:
            if a < b:
                o = random.choice([0,0,1,2,3,4]) * random.random()
                overlap[a][b] = o; overlap[b][a] = o

    init_pos = random_init_positions()
    return {'activity':activity, 'dominance':dominance,
            'overlap':overlap, 'init_pos':init_pos,
            'n_active':n_active}


In [ ]:
# ------------------------------------------------------------
# Cell 6 — Build ONE snapshot (UPDATED: fix 7 normalization,
#          consistent EPS, hard_case metadata)
# ------------------------------------------------------------
def normalize_edge_features(d_az, d_el, spec, rel_db, dom_r, ovf, ovd):
    """Fix 7: ensure ALL edge features lie in [-1,1] or [0,1].
    Attention is scale-sensitive — unnormalized dB would dominate."""
    d_az  = float(np.clip(d_az, -1, 1))      # already normalized pos diff
    d_el  = float(np.clip(d_el, -1, 1))
    spec  = float(np.clip((spec + 1)/2, 0, 1))  # cosine [-1,1] -> [0,1]
    rel_db= float(np.clip(rel_db, -1, 1))    # rms diff, already [-1,1]
    dom_r = float(np.clip(dom_r, 0, 1))      # sigmoid output
    ovf   = float(np.clip(ovf, 0, 1))
    ovd   = float(np.clip(ovd, 0, 1))
    return [d_az, d_el, spec, rel_db, dom_r, ovf, ovd]


def build_snapshot(args):
    seed, = args
    random.seed(seed); np.random.seed(seed)

    cfg = sample_scene_config()
    clips, spk_ids, hard = pick_clips_for_scene()
    az_deg = cfg['init_pos'][:,0]*180.0
    el_deg = cfg['init_pos'][:,1]*90.0
    hrtf = random.choice(HRTFS)

    embeddings = np.zeros((N_SPK,128), dtype=np.float32)
    rms = np.zeros(N_SPK, dtype=np.float32)

    for k in range(N_SPK):
        mono = load_clip(clips[k])
        if cfg['activity'][k] == 0:
            mono = mono * 0.01                       # silent
        mono = mono * (0.3 + 0.7*cfg['dominance'][k])  # loudness ~ dominance
        rms[k] = np.sqrt(np.mean(mono**2))
        stereo = hrtf_convolve(mono, hrtf, az_deg[k], el_deg[k])
        with torch.no_grad():
            embeddings[k] = cnn(torch.tensor(stereo).unsqueeze(0)).squeeze(0).numpy()

    rms_norm = rms/(rms.max() + EPS)

    # Node features [4,133]
    nodes = np.concatenate([
        embeddings, rms_norm[:,None], cfg['dominance'][:,None],
        cfg['activity'][:,None], cfg['init_pos'][:,0:1],
        cfg['init_pos'][:,1:2],
    ], axis=1).astype(np.float32)

    # Edges [2,12] + normalized features [12,7]
    src,dst,efeat = [],[],[]
    ovl_norm = np.clip(cfg['overlap']/OVERLAP_WIN, 0, 1)
    for i in range(N_SPK):
        for j in range(N_SPK):
            if i==j: continue
            src.append(i); dst.append(j)
            d_az = cfg['init_pos'][i,0]-cfg['init_pos'][j,0]
            d_el = cfg['init_pos'][i,1]-cfg['init_pos'][j,1]
            spec = np.dot(embeddings[i],embeddings[j])/(
                   np.linalg.norm(embeddings[i])*np.linalg.norm(embeddings[j])+EPS)
            rel_db = rms_norm[i]-rms_norm[j]
            dom_r = 1/(1+np.exp(-ALPHA_SIG*(cfg['dominance'][i]-cfg['dominance'][j])))
            ovf = float(cfg['activity'][i]==1 and cfg['activity'][j]==1)
            ovd = ovl_norm[i][j]
            efeat.append(normalize_edge_features(d_az,d_el,spec,rel_db,dom_r,ovf,ovd))

    return {
        'x': torch.tensor(nodes),
        'edge_index': torch.tensor([src,dst],dtype=torch.long),
        'edge_attr': torch.tensor(efeat,dtype=torch.float32),
        'prev_positions': torch.tensor(cfg['init_pos']),
        'activity_mask': torch.tensor(cfg['activity']),
        'dominance': torch.tensor(cfg['dominance']),
        'overlap_duration': torch.tensor(cfg['overlap']),
        'embeddings': torch.tensor(embeddings),
        # Fix 9: metadata for debugging / ablation / demo
        'meta': {'seed':seed, 'speaker_ids':spk_ids,
                 'hard_case':hard, 'n_active':int(cfg['n_active'])},
    }


In [ ]:
# Cell 7 — Generate dataset with multiprocessing Pool
# NOTE: multiprocessing on Colab — CNN + HRTFS must be picklable/global.
# Each worker re-imports globals. For Colab safety we use a chunked loop
# with Pool over seeds.

N_SCENES = 6000   # adjust: 5k-10k
BATCH    = 500    # save every BATCH snapshots

def generate(n_scenes=N_SCENES, batch=BATCH, n_workers=None):
    n_workers = n_workers or max(2, cpu_count()-1)
    t0=time.time(); saved=0
    for b_start in range(0, n_scenes, batch):
        seeds = [(b_start+i,) for i in range(min(batch, n_scenes-b_start))]
        with Pool(n_workers) as pool:
            snaps = pool.map(build_snapshot, seeds)
        fn = OUT_DIR + f'snapshots_{b_start:06d}.pt'
        torch.save(snaps, fn)
        saved += len(snaps)
        el=time.time()-t0
        print(f'  saved {saved}/{n_scenes}  ({el:.0f}s, '
              f'{saved/el:.1f} scenes/s)')
    print(f'DONE. {saved} scenes in {time.time()-t0:.0f}s')

generate()

In [ ]:
# Cell 8 — Verify generated dataset
files = sorted(Path(OUT_DIR).glob('snapshots_*.pt'))
total=0
for f in files:
    d=torch.load(f); total+=len(d)
print(f'{len(files)} files, {total} total snapshots')

# Sanity check one
d=torch.load(files[0]); s=d[0]
print('keys:', list(s.keys()))
print('x', s['x'].shape, 'edge_attr', s['edge_attr'].shape)
print('spectral_corr range:',
      s['edge_attr'][:,2].min().item(), s['edge_attr'][:,2].max().item())
print('activity:', s['activity_mask'])

with open(OUT_DIR+'dataset_meta.json','w') as fp:
    json.dump({'total':total,'files':len(files),
               'node_dim':NODE_DIM,'edge_dim':EDGE_DIM}, fp, indent=2)
print('\nDay 9 complete. Ready for Day 10 training.')